# 30 – LODES Spatial Units (Blocks / Block Groups) Cleaning

This notebook prepares geometries for **LODES jobs data**, matching the spatial unit used in the LODES table (blocks or block groups).

**Goals:**
- Load the corresponding block / block-group shapefile.
- Standardize CRS to project CRS.
- Keep only the GEOID and geometry.
- Save a cleaned layer ready to join with LODES job counts.

In [1]:
from pathlib import Path
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
JOBS_DIR = DATA_DIR / "jobs"

JOBS_DIR

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs')

In [2]:
raw_lodes_geom_path = JOBS_DIR / "lodes_geometry" / "AZ_blck_grp_2023.shp"
lodes_geom_clean_path = JOBS_DIR / "lodes_processed" / "lodes_units_clean.gpkg"

raw_lodes_geom_path, lodes_geom_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_geometry/AZ_blck_grp_2023.shp'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/lodes_units_clean.gpkg'))

In [3]:
lodes_units = gpd.read_file(raw_lodes_geom_path)

print("LODES spatial features:", len(lodes_units))
print("CRS:", lodes_units.crs)

lodes_units.head()

LODES spatial features: 4773
CRS: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",37.5],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102003"]]


,GISJOIN,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOID,GEOIDFQ,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,Shape_Leng,Shape_Area,ORIG_FID,geometry
0,G04000109426001,04,001,942600,1,040019426001,1500000US040019426001,Block Group 1,G5030,S,1.094120e+09,208117.0,201666.006696,1.094328e+09,98943,"POLYGON ((-1205442.684 24822.029, -1205445.87 ..."
1,G04000109426002,04,001,942600,2,040019426002,1500000US040019426002,Block Group 2,G5030,S,4.311312e+08,267520.0,97759.964618,4.313987e+08,99020,"POLYGON ((-1204504.841 30811.982, -1204498.821..."
2,G04000109427001,04,001,942700,1,040019427001,1500000US040019427001,Block Group 1,G5030,S,4.503639e+08,754604.0,125325.976779,4.511185e+08,99132,"POLYGON ((-1146477.066 22667.315, -1146479.085..."
3,G04000109427002,04,001,942700,2,040019427002,1500000US040019427002,Block Group 2,G5030,S,1.160417e+09,346444.0,335124.099548,1.160764e+09,99130,"POLYGON ((-1170443.973 26051.139, -1170468.175..."
4,G04000109427003,04,001,942700,3,040019427003,1500000US040019427003,Block Group 3,G5030,S,5.312418e+08,261769.0,161221.806338,5.315037e+08,99133,"POLYGON ((-1170140.302 21362.1, -1170059.64 21..."


In [ ]:
PROJECT_CRS = "EPSG:2868"

if PROJECT_CRS is not None and lodes_units.crs != PROJECT_CRS:
    lodes_units = lodes_units.to_crs(PROJECT_CRS)
    print("Reprojected to PROJECT_CRS:", PROJECT_CRS)
else:
    print("No reprojection performed (check CRS).")

Reprojected to PROJECT_CRS: EPSG:2868


In [5]:
lodes_units.columns

Index(['GISJOIN', 'STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE', 'GEOID',
       'GEOIDFQ', 'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER',
       'Shape_Leng', 'Shape_Area', 'ORIG_FID', 'geometry'],
      dtype='object')

In [7]:
lodes_units_maricopa = lodes_units[
    (lodes_units["STATEFP"] == "04") &
    (lodes_units["COUNTYFP"] == "013")
].copy()

print("Original BG count:", len(lodes_units))
print("Maricopa BG count:", len(lodes_units_maricopa))

Original BG count: 4773
Maricopa BG count: 2806


In [9]:
lodes_units_clean = lodes_units_maricopa[[
    "GEOID",
    "STATEFP",
    "COUNTYFP",
    "TRACTCE",
    "BLKGRPCE",
    "geometry"
]].copy()

In [10]:
lodes_units_clean.to_file(lodes_geom_clean_path, driver="GPKG")
print("Saved cleaned LODES spatial units to:", lodes_geom_clean_path)

Saved cleaned LODES spatial units to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\jobs\lodes_processed\lodes_units_clean.gpkg
